# Robot and twin observation

Build an offline robot factory from ordered instructions. Start with one fixed pickup fixture and one fixed destination. Pair roles: driver edits; observer predicts and checks; swap halfway. Only the instructor enables supervised hardware.

Simulation, paper, reports and measured physical outcomes are separate evidence. Never claim a grasp or placement from a command log alone. Unknown or stale observations require pause and inspection.

Save a pair copy. Run cells from the top with Shift+Enter. The essential route works offline. Read the explicit evidence and capture-mode labels below.

| Minutes | Duration | Session |
|---|---|---|
| 0-20 | 20 | Meet our factory |
| 20-45 | 25 | Ordered instructions |
| 45-65 | 20 | Observations and uncertainty |
| 65-95 | 30 | First Python edits |
| 95-105 | 10 | Break |
| 105-120 | 15 | Teach inspect and replay demonstration |
| 120-165 | 45 | Explore three stations |
| 165-210 | 45 | Build the physical sequence |
| 210-230 | 20 | Improve one failure |
| 230-240 | 10 | Three pair demonstrations |

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "workshop").is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Read-only observation notebook. Capture mode is selected below.")


A joint is a movable connection. A digital twin is a screen model updated from measurements. Predict an angle change, observe the follower and compare its screen model. This notebook only observes: it never enables, changes modes or sends motion. The instructor owns the RViz controls. Keep clear of the moving workspace and never move an enabled follower by hand.

15-minute observation rotation. Worked example, predict/change/run/observe/explain and success criteria are on the teach demonstration mission card.

## Open and read

3 minutes. Save a pair copy. Run from the top. Default DEMO is clearly labelled invented data. The prepared instructor launch `tools/launch_live_workshop.py --live --ros-wrapper /path/to/prepared/wrapper` sets `WORKSHOP_LIVE=1` and launches a bounded read-only capture through the existing ROS wrapper. Do not install ROS in Jupyter. Predict whether all joint angles will be equal.

In [ ]:
import os
from workshop.live_observer import observe, plot_samples, JOINTS
LIVE = os.environ.get("WORKSHOP_LIVE") == "1"
print("LIVE topic observation: read-only" if LIVE else "DEMO: invented data, no robot measurements")


Ask the instructor to hold the leader still. Every capture takes a new three-second sample and finishes its observer process. LIVE capture failure produces no measurements and an explicit unavailable message; it never switches to DEMO. To choose offline intentionally, restart via the launcher without --live.

In [ ]:
report = observe(live=LIVE, duration=3)
print(report["mode"])
print(report["samples"][-1] if report["samples"] else report.get("error", "No samples"))


Read fresh, stale and not_running separately. Fresh requires recent receipt and source timestamps. Stale means too old; not_running means no message received and may mean incorrect discovery settings. Never interpret a missing angle as zero or permission to move. The follower header marks publication, not independent motor acquisition. An old latched teleop status may describe a disabled or previous state.

## Pick one joint

3 minutes. Predict which line will be higher. Change only JOINT to another printed name and run again. The helper converts five body joint angles from radians to degrees. The gripper JointState value is a visual estimate, so it is excluded. A full turn is 360 degrees or about 6.28 radians.

In [ ]:
print(JOINTS)
JOINT = "shoulder_pan"
plot_samples(report, JOINT)


## Predict one small change

4 minutes. Relative mode requests follower start + leader now - leader start. Absolute mode requests the calibrated leader angle. These next invented numbers are paper predictions, including during LIVE observation. Change only LEADER_NOW after predicting. No arithmetic here reaches the controller.

In [ ]:
LEADER_START = 10
FOLLOWER_START = 30
LEADER_NOW = 15
print("DEMO: invented angles; no robot measurement")
print("Relative prediction:", FOLLOWER_START + LEADER_NOW - LEADER_START, "degrees")
print("Absolute prediction:", LEADER_NOW, "degrees")


The original prediction is relative 35 degrees and absolute 15 degrees. Only in the prepared LIVE station, the instructor chooses a safe joint and demonstrates one small movement using RViz. Predict direction first. The controller still enforces its limits and STOP. In DEMO discuss the invented trace. Do not change modes while moving.

In [ ]:
report = observe(live=LIVE, duration=3)
print(report["mode"])
print(report.get("error", "Read fresh/stale labels; a request is not a measured result."))
plot_samples(report, JOINT)


## Explain the twin

3 minutes. Compare one fresh joint direction with RViz. A model driven by the follower stream is not an independent sensor. Different leader/follower angles can be correct in relative mode because start positions differ. Delay, stale data, reference errors or a wrong model can also cause mismatch; a graph alone cannot identify the cause. Read only the last fresh teleop status below. Missing or stale status requires the instructor.

In [ ]:
statuses = [s["teleop"]["value"] for s in report["samples"]
            if s.get("teleop", {}).get("state") == "fresh"]
print(statuses[-1] if statuses else "No fresh teleop status. Ask instructor.")


## Record and rotate

2 minutes. Record selected joint, predicted relative/absolute angle, observed direction, DEMO or LIVE evidence, and one reason the graph and twin might disagree. Save and tell the next pair which evidence source you used. If LIVE is unavailable, record that failure and ask the instructor. An explicit restart without --live is required before using invented offline data.